# Model comparison

Runs every saved model on the same 624 test X-rays and compares them side by side. Each model gets its own preprocessing, so differences come from the models, not the setup.

To add a new version, train it, then add one entry to `MODELS` below.

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, roc_curve, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay)

class_names = ['NORMAL', 'PNEUMONIA']

## Models

By default each version uses its **newest** saved file in `models/`, and the cell prints which files were picked. Each entry gives the saved file, a function that turns one image path into the model's input, and how to read a pneumonia probability out of the model's output (V2 has a 2-class softmax, V1 and V3 a single sigmoid).

V1 was trained on raw 0–255 pixel values (its `/255` scaling was only applied to an earlier dataset, not the one used for training), so it is given unscaled images here.

In [ ]:
def v1_input(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, (256, 256)).numpy()

def v2_input(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (196, 196))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) / 255.0
    return img[..., None]

def v3_input(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, (224, 224)).numpy()   # DenseNet preprocessing happens inside the model

def latest(prefix):
    """Newest saved model for a version, e.g. latest('v1_cnn') -> models/v1_cnn_2026-10-12.keras.
    File names end in an ISO date, so sorting them alphabetically also sorts them by date."""
    paths = sorted(glob.glob(os.path.join('models', f'{prefix}_*.keras')))
    if not paths:
        raise FileNotFoundError(f'No saved model matching models/{prefix}_*.keras')
    return paths[-1]

# To compare a specific older file instead, replace latest(...) with its path, e.g. 'models/v1_cnn_2025-10-01.keras'
MODELS = {
    'V1 CNN':         {'path': latest('v1_cnn'),         'input': v1_input, 'prob': lambda out: out.ravel()},
    'V2 CNN':         {'path': latest('v2_cnn'),         'input': v2_input, 'prob': lambda out: out[:, 1]},
    'V3 DenseNet121': {'path': latest('v3_densenet121'), 'input': v3_input, 'prob': lambda out: out.ravel()},
}

for name, spec in MODELS.items():
    print(f'{name:<15} {spec["path"]}')

In [ ]:
test_df = pd.DataFrame(
    [(p, label) for label, name in enumerate(class_names)
     for p in sorted(glob.glob(os.path.join('data', 'test', name, '*.jpeg')))],
    columns=['image', 'label'])
y_test = test_df['label'].values

probs = {}
for name, spec in MODELS.items():
    model = tf.keras.models.load_model(spec['path'], compile=False)
    X = np.stack([spec['input'](p) for p in test_df['image']])
    probs[name] = spec['prob'](model.predict(X, batch_size=32, verbose=0))
    print(f'{name}: {len(probs[name])} predictions')

## Metrics

- **Sensitivity** (pneumonia recall): share of pneumonia cases caught. Missing one is the costly error.
- **Specificity** (normal recall): share of healthy X-rays correctly cleared. This was V2's weak point.
- **ROC-AUC / PR-AUC** don't depend on a threshold: they measure how well the model ranks pneumonia above normal.
- **Specificity at 95% sensitivity**: if each model's threshold is set so it catches 95% of pneumonia cases, how many healthy X-rays does it clear? This compares the models at the same safety level, so one model can't look better just by using a different threshold.

In [ ]:
def specificity_at_sensitivity(y, p, target=0.95):
    fpr, tpr, _ = roc_curve(y, p)
    return 1 - fpr[np.argmax(tpr >= target)]

rows = {}
for name, p in probs.items():
    y_pred = (p >= 0.5).astype(int)
    rows[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Sensitivity (pneumonia recall)': recall_score(y_test, y_pred),
        'Specificity (normal recall)': recall_score(y_test, y_pred, pos_label=0),
        'Precision': precision_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, p),
        'PR-AUC': average_precision_score(y_test, p),
        'Specificity @ 95% sensitivity': specificity_at_sensitivity(y_test, p),
    }

results = pd.DataFrame(rows)
names = list(probs)
if len(names) >= 2:
    results['Change'] = results[names[-1]] - results[names[-2]]
results.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for name, p in probs.items():
    RocCurveDisplay.from_predictions(y_test, p, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, p, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], 'k--', label='Chance')
axes[0].legend()
axes[0].set_title('ROC curve (test)')
axes[1].set_title('Precision-recall curve (test)')
plt.show()

fig, axes = plt.subplots(1, len(probs), figsize=(6 * len(probs), 5))
for ax, (name, p) in zip(np.atleast_1d(axes), probs.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, (p >= 0.5).astype(int), display_labels=class_names,
                                            cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(f'{name} (threshold 0.5)')
plt.show()

## Is the improvement real?

A better score on 624 images could be luck. Two checks compare the last two models in `MODELS`:

- **McNemar's test** looks only at the X-rays the models disagree on. If the newer model fixes many more of the older one's mistakes than it introduces, the p-value is small (below 0.05 is the usual cut-off).
- **Bootstrap confidence intervals:** the test set is resampled 2,000 times to see how much the accuracy and ROC-AUC gaps vary. If the 95% interval doesn't include 0, the improvement is consistent.

In [ ]:
old, new = names[-2], names[-1]
old_correct = (probs[old] >= 0.5).astype(int) == y_test
new_correct = (probs[new] >= 0.5).astype(int) == y_test

fixed = int(np.sum(~old_correct & new_correct))
broken = int(np.sum(old_correct & ~new_correct))
p_value = binomtest(fixed, fixed + broken, 0.5).pvalue if fixed + broken else 1.0
print(f'{new} vs {old}')
print(f'  X-rays {new} fixed: {fixed}')
print(f'  X-rays {new} got wrong that {old} had right: {broken}')
print(f"  McNemar's exact test p-value: {p_value:.2g}")

rng = np.random.default_rng(42)
acc_diffs, auc_diffs = [], []
for _ in range(2000):
    idx = rng.integers(0, len(y_test), len(y_test))
    acc_diffs.append(new_correct[idx].mean() - old_correct[idx].mean())
    auc_diffs.append(roc_auc_score(y_test[idx], probs[new][idx]) - roc_auc_score(y_test[idx], probs[old][idx]))

for label, diffs in [('Accuracy', acc_diffs), ('ROC-AUC', auc_diffs)]:
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f'  {label} change: {np.mean(diffs):+.3f}  (95% CI {lo:+.3f} to {hi:+.3f})')

## Where the models disagree

X-rays that one model gets right and the other gets wrong, with each model's pneumonia probability. Useful for spotting what kind of image the newer model has learned to handle.

In [ ]:
def show_examples(mask, title, n=8):
    idx = np.where(mask)[0][:n]
    if len(idx) == 0:
        print(f'{title}: none')
        return
    plt.figure(figsize=(3 * len(idx), 3.8))
    for i, j in enumerate(idx):
        plt.subplot(1, len(idx), i + 1)
        plt.imshow(cv2.imread(test_df['image'][j], cv2.IMREAD_GRAYSCALE), cmap='gray')
        plt.title(f'True: {class_names[y_test[j]]}\n{old}: {probs[old][j]:.2f}\n{new}: {probs[new][j]:.2f}', fontsize=8)
        plt.axis('off')
    plt.suptitle(title)
    plt.show()

show_examples(~old_correct & new_correct, f'Fixed by {new}')
show_examples(old_correct & ~new_correct, f'Broken by {new}')